<div style="background-color:#3B4252; border-left: 6px solid #88C0D0; padding: 16px 20px; border-radius: 6px; font-family: sans-serif;">

# 🚀 Practical Component Instructions

- **Just watching?** Follow along as we walk though the code together.
- **Want to run the code?** Click **File > Save a copy in Drive/GitHub** at the top left. This will give you your own private sandbox to run the R code, generate the contents, and make edits.

</div>

---

# Statistical Methods for Linguistic Data
## CLA 2026 Crash Course Practical Component

This notebook accompanies the workshop lecture. We will work through five analyses using synthetic data from a synthetic study on the focus sensitive particle (FSP) *only* in English.

---

### Synthetic study 1: Experimental study

**Research question:** What effects do we observe in the different structural positions of the focus sensitive particle *only*?

**Participants:** 83 adult native English speakers recruited from a university population

**Design:** 2 x 4 factorial
- **Context** (2 levels): *match* (sentence is TRUE given context) vs. *mismatch* (sentence is FALSE given context)
- **FP position** (4 levels): the position of *only* in the sentence
  - `FSP-S-V-O`: *Only Dale drinks coffee.*
  - `S-FSP-V-O`: *Dale only drinks coffee.*
  - `S-V-FSP-O`: *Dale drinks only coffee.*
  - `S-V-O-FSP`: *Dale drinks coffee only.*

**Three tasks:**

| Task | Measure | Outcome variable | Type |
|------|---------|-----------------|------|
| Sentence production | Peak F0 on focused constituent (Hz) | `peak_f0` | Continuous |
| Acceptability judgment | 1-7 Likert rating | `rating` | Ordinal |
| Truth Value Judgment Task (TVJT) | Accuracy (correct/incorrect) | `correct` | Binary |
| TVJT | Response time (ms) | `rt_ms` | Continuous |

### Synthetic study 2: Corpus study

**Research question:** Does the frequency of FSP position types vary across registers?

**Data:** Occurrence counts of *only* in each FSP position, drawn from 160 documents across 8 corpora representing four registers:

| Corpus type | Register | Example corpora |
|-------------|----------|-----------------|
| Newspaper | Written formal | The National Chronicle, The Daily Register |
| Social media written | Written informal | RedditLing, BlueSky Academic |
| Legal proceedings | Spoken formal | Federal Court Transcripts, Provincial Court Transcripts |
| Social media spoken | Spoken informal | LinguistTok, GrammarReels |

Each document contributes 4 rows (i.e., one count per FSP position) giving 640 rows total. Document length varies substantially across corpus types which must be accounted for in modelling.

**Outcome variable:** `count` (non-negative integer); data are overdispersed, requiring negative binomial regression rather than Poisson.

---

# 0. Setup

Install and load all required packages. **Run this cell first**; it may take a minute or two on a fresh run.

In [1]:
# Install pacman if not already available, then use it to load everything else
if (!require("pacman")) install.packages("pacman")

pacman::p_load(
  lme4, # linear and generalized linear mixed effects models
  lmerTest, # p-values and degrees of freedom for lmer()
  ggplot2, # plotting
  dplyr, # data wrangling
  MASS, # negative binomial
  ordinal, # cumulative link models (ordinal regression)
  patchwork, # combining multiple plots
  viridis # visually-impaired-friendly colour palette
)

cat("All packages successfully loaded!")

Loading required package: pacman



All packages successfully loaded!

---
# -------------------- Synthetic study #1: Experimental study --------------------

## 1. Load, inspect, and format the data

There are four datasets for study #1 hosted on GitHub. Run the cell below to load them all at once.

In [ ]:
# Load all four datasets directly from GitHub

participants <- read.csv("https://raw.githubusercontent.com/lhracs/CLA2026_crash_course/refs/heads/main/participants_synth.csv",
                         stringsAsFactors = FALSE)
production <- read.csv("https://raw.githubusercontent.com/lhracs/CLA2026_crash_course/refs/heads/main/production_synth.csv",
                         stringsAsFactors = FALSE)
acceptability <- read.csv("https://raw.githubusercontent.com/lhracs/CLA2026_crash_course/refs/heads/main/acceptability_synth.csv",
                          stringsAsFactors = FALSE)
tvjt <- read.csv("https://raw.githubusercontent.com/lhracs/CLA2026_crash_course/refs/heads/main/tvjt_synth.csv",
                 stringsAsFactors = FALSE)

cat("All datasets successfully loaded!\n")
cat("  participants:", nrow(participants), "rows\n")
cat("  production:", nrow(production), "rows\n")
cat("  acceptability:", nrow(acceptability),"rows\n")
cat("  tvjt:", nrow(tvjt), "rows\n")

Review the structure of each dataset. Variable type information helps you determine what operations you can perform on a particular variable and, to an extent, dictates which statistical tests are valid.

In [ ]:
# Quick look at each dataset and check variable types

head(participants) # preview first 6 rows of an object
str(participants) # get structure of an object

head(production)
str(production)

head(acceptability)
str(acceptability)

head(tvjt)
str(tvjt)

> Key question before modelling: **What type is each outcome variable?**
>
> - `peak_f0`: continuous (Hz)
> - `rating`: ordinal (1-7 Likert scale, NOT continuous)
> - `correct`: binary (0 or 1)
> - `rt_ms`: continuous, but likely right-skewed (reaction times)
>
> Each type requires a different model family. This is the first decision in the analysis pipeline.

In [ ]:
# Set factor levels so models use a consistent reference level

# Reference: FSP-S-V-O (focus sensitive particle directly precedes subject)
production$fsp_position <- factor(production$fsp_position,
                                  levels = c("FSP-S-V-O","S-FSP-V-O","S-V-FSP-O","S-V-O-FSP"))
acceptability$fsp_position <- factor(acceptability$fsp_position,
                                    levels = c("FSP-S-V-O","S-FSP-V-O","S-V-FSP-O","S-V-O-FSP"))
tvjt$fsp_position <- factor(tvjt$fsp_position,
                            levels = c("FSP-S-V-O","S-FSP-V-O","S-V-FSP-O","S-V-O-FSP"))

# Set context reference level to mismatch
production$context <- factor(production$context, levels = c("mismatch","match"))
acceptability$context <- factor(acceptability$context, levels = c("mismatch","match"))
tvjt$context <- factor(tvjt$context, levels = c("mismatch","match"))

# Rating must be an ordered factor for ordinal regression
acceptability$rating <- factor(acceptability$rating, levels = 1:7, ordered = TRUE)

cat("Factor levels set!\n")

---

## 2. Visualize before you model

Always plot your data before fitting any model. Summary statistics can be deeply misleading.
We will look at each outcome variable in turn.

### 2.1 Distribution of RT: Histogram

In [ ]:
# Histogram of raw RT values

ggplot(tvjt, aes(x = rt_ms)) +
  geom_histogram(bins = 40, fill = viridis(1, option = "D"), color = "#2E3440", alpha = 0.85) +
  geom_vline(aes(xintercept = mean(rt_ms)), color = viridis(1, option = "A"),
             linewidth = 1, linetype = "dashed") +
  labs(
    title = "Distribution of response times",
    subtitle = "Dashed line = mean. Notice the right skew and outliers.",
    x = "RT (ms)",
    y = "Count"
  ) +
  theme_minimal(base_size = 13)

> **What to notice:**
> - The distribution is right-skewed; a long tail of slow responses
> - The mean (dashed line) is pulled toward the tail
> - A few extreme outliers are visible on the right; could consider removing these (e.g., using the Tukey method / interquartile range method  
>
> This tells us we should log-transform RT before fitting a linear model.

### 2.2 Ratings by FSP position and context: Violin + boxplot

In [ ]:
# Violin plot with overlaid boxplot for ratings by condition

ggplot(acceptability, aes(x = fsp_position, y = as.numeric(rating), fill = context)) +
  geom_violin(position = position_dodge(0.8), alpha = 0.7) +
  geom_boxplot(position = position_dodge(0.8), width = 0.2,
               outlier.shape = 21, outlier.size = 1.5) +
  scale_fill_viridis_d(option = "D", begin = 0.3, end = 0.75) + # applies a discrete viridis palette
  labs(
    title = "Acceptability ratings by FSP position and context",
    subtitle = "1 = very unacceptable, 7 = very acceptable",
    x = "FSP position",
    y = "Rating (1-7)",
    fill = "Context"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top")

> **What to notice:**
> - Match contexts are rated higher than mismatch contexts across all FSP positions
> - `S-FSP-V-O` (preverbal *only*) shows the most spread and the lowest match ratings
> - Ratings cluster at the high end in match conditions; ceiling effects are visible
> - This is why we need ordinal regression: the distributions are not normal, and the gaps between levels are not equal

### 2.3 F0 by FSP position: Strip chart

In [ ]:
# Strip chart of peak F0 by FSP position, overlaid with group means and SE

ggplot(production, aes(x = fsp_position, y = peak_f0, colour = fsp_position)) +
  geom_jitter(width = 0.2, alpha = 0.35, size = 1.5) +
  stat_summary(fun = mean, geom = "crossbar", width = 0.5,
               colour = "#2E3440", linewidth = 0.8) +
  stat_summary(fun.data = mean_se, geom = "errorbar", width = 0.25,
               colour = "#2E3440", linewidth = 0.8) +
  scale_colour_viridis_d(option = "D") +
  labs(
    title = "Peak F0 on focused constituent by FSP position",
    subtitle = "Crossbar = mean +/- SE. Individual observations shown as points.",
    x = "FP position",
    y = "Peak F0 (Hz)"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")

> **What to notice:**
> - There is a clear gradient: `FSP-S-V-O` shows the highest F0, `S-V-O-FSP` the lowest
> - Individual variation is large; some participants have much higher baseline F0
> - A few outlier points are visible at the extremes
> - The group pattern is consistent despite individual differences; this is what random effects model

### 2.4 Accuracy by FSP position and context: Bar chart

In [ ]:
# Mean accuracy by FSP position and context
tvjt_summary <- tvjt %>%
  group_by(fsp_position, context) %>%
  summarise(mean_acc = mean(correct), se = sd(correct) / sqrt(n()), .groups = "drop")

ggplot(tvjt_summary, aes(x = fsp_position, y = mean_acc, fill = context)) +
  geom_col(position = position_dodge(0.7), width = 0.6) +
  geom_errorbar(aes(ymin = mean_acc - se, ymax = mean_acc + se),
                position = position_dodge(0.7), width = 0.25) +
  geom_hline(yintercept = 0.5, linetype = "dashed", colour = "#4C566A") +
  scale_fill_viridis_d(option = "D", begin = 0.3, end = 0.75) +
  scale_y_continuous(limits = c(0, 1), labels = scales::percent) +
  labs(
    title = "TVJT accuracy by FSP position and context",
    subtitle = "Dashed line = chance (50%). Error bars = +/- 1 SE.",
    x = "FP position",
    y = "Mean accuracy",
    fill = "Context"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top")

> **What to notice:**
> - All conditions are well above chance; participants can assign truth values to sentences with *only*
> - Accuracy drops for mismatch conditions, especially `S-FSP-V-O`
> - `S-FSP-V-O` mismatch is the hardest condition; consistent with a preverbal ambiguity hypothesis
> - We will model this with logistic regression in Section 5

---

## 3. Linear mixed effects model: Peak F0

**Outcome:** `peak_f0` (continuous, Hz)
**Question:** Does FSP position predict where pitch accent lands?

We use `lmer()` from the `lme4` package, with random intercepts for participants and items.

In [ ]:
# Fit linear mixed effects model for F0
# Random intercepts for participant and item
model_f0 <- lmer(peak_f0 ~ fsp_position + context + (1 | participant_id) + (1 | item_id),
                 data = production,
                 REML = TRUE)

summary(model_f0)

> **Reading the output:**
> - **Fixed effects:** coefficients are in Hz, relative to the reference level (`FSP-S-V-O`, mismatch)
> - **Intercept:** predicted F0 for the reference condition
> - **fsp_positionS-FSP-V-O:** how much lower F0 is for preverbal FSP sentences compared to reference
> - **Random effects:** variance due to participants vs. items; people differ more than stimuli do
> - **t value:** lmerTest provides degrees of freedom and p-values
---
> **Random effects**
> - Participant variance (457.18, SD = 21.4 Hz): participants vary substantially in their baseline F0. This is expected -- some people just have higher voices. The model is correctly absorbing this
> - Item variance (36.87, SD = 6.1 Hz): items vary less than participants, also expected; the stimuli were well controlled
> - Residual variance (150.82, SD = 12.3 Hz): leftover noise after accounting for fixed effects and random effects
---
> **Fixed effects**
> Reference level: FSP-S-V-O, mismatch
> - Intercept (210.3 Hz): predicted peak F0 for the reference condition (FSP-S-V-O, mismatch context). This is the baseline.
> - S-FSP-V-O (-24.7 Hz, p < .001): preverbal FSP sentences have peak F0 about 25 Hz lower than FSP-initial sentences
> - S-V-FSP-O (-27.3 Hz, p < .001): post-object FSP sentences have peak F0 about 27 Hz lower than FSP-initial
> - S-V-O-FSP (-36.6 Hz, p < .001): sentence-final FSP sentences have the lowest F0, about 37 Hz lower than FSP-initial. This is the largest effect; the gradient across positions is clear
> - contextmatch (+8.3 Hz, p < .001): match contexts produce about 8 Hz higher peak F0 than mismatch contexts

In [ ]:
# Confidence intervals for fixed effects
confint(model_f0, method = "Wald")

> **A note on NA sigma values**

> .sigma is the residual standard deviation (the leftover variability not explained by the fixed effects or random effects; in other words, the noise in the model). `NA` values are expected, not an error. Wald confidence intervals assume an estimate could
> plausibly fall anywhere on a number line, but standard deviations cannot be negative.
> Rather than return intervals that include impossible values, R returns `NA`.
> The fixed effects intervals above are unaffected and can be interpreted normally.
> If you need intervals for the variance components, use `confint(model, method = "profile")`.

In [ ]:
# Check model assumptions; residuals vs. fitted
plot(model_f0, main = "Residuals vs. Fitted for F0 model")

In [ ]:
# Q-Q plot of residuals
qqnorm(resid(model_f0), main = "Q-Q plot for F0 model residuals")
qqline(resid(model_f0), col = viridis(1, option = "A"))

> **What to check:**
> - Residuals vs fitted: should look like random scatter around zero -- no fan or curve pattern
> - Q-Q plot: points should fall close to the diagonal line
> - If assumptions are met, we can trust our coefficients and p-values

In [ ]:
# Effect sizes Cohen's d for fixed effects
# Standardize each coefficient by the residual standard deviation

residual_sd <- sigma(model_f0)  # extracts residual SD directly from the model
coefs <- fixef(model_f0)

cat("Cohen's d for fixed effects:\n\n")
for (nm in names(coefs)) {
  d <- coefs[nm] / residual_sd
  cat(sprintf("  %-30s d = %6.3f\n", nm, d))
}

# d <= 0.2 is small
# d = 0.5 is moderate/medium
# d >= 0.8 is large

---

## 4. Ordinal regression: Acceptability ratings

**Outcome:** `rating` (ordered, 1-7)
**Question:** Does FSP position and context affect acceptability?

### Why not lmer()?

Before fitting the correct model, we should look at why treating ratings as continuous is a problem.

In [ ]:
# What happens if we treat rating as continuous and use lmer()?
# Convert rating back to numeric for demonstration persons
acceptability$rating_num <- as.numeric(acceptability$rating)

incorrect_mod <- lmer(rating_num ~ fsp_position + context + (1 | participant_id) + (1 | item_id),
                      data = acceptability, REML = TRUE)

summary(incorrect_mod)

In [ ]:
# Check the residuals
qqnorm(resid(incorrect_mod), main = "Q-Q plot for incorrect model (lmer on Likert ratings)")
qqline(resid(incorrect_mod), col = viridis(1, option = "A"))

> **Q-Q plot interpretation:** Points should fall on the diagonal if residuals are normally
> distributed. The deviation at both ends indicates the residuals are not normal which is expected
> when a bounded ordinal outcome is forced into a linear model.

In [ ]:
# Show that lmer predicts impossible values
range(fitted(incorrect_mod))

> **Impossible predictions:** The range above shows fitted values outside the 1-7 scale.
> A rating of 7.15 cannot exist in your data. A model that predicts impossible values
> is misspecified regardless of whether the p-values look significant.

In [ ]:
# Show the actual distribution of ratings
ggplot(acceptability, aes(x = rating)) +
  geom_bar(fill = viridis(1, option = "D")) +
  facet_wrap(~ fsp_position + context) +
  labs(title = "Distribution of ratings by condition") +
  theme_minimal(base_size = 12)

> **Distribution of ratings:** Ratings cluster at discrete values and do not spread
> smoothly across the scale the way a continuous variable should. The floor and ceiling effects
> visible here (clustering toward ends of the scale) are additional evidence that a linear model is
> inappropriate. Ordinal regression models these discrete categories directly.

In [ ]:
# Simulate two conditions with identical means but different distributions
set.seed(8675309)
demo <- data.frame(
  condition = rep(c("A", "B"), each = 50),
  rating = c(
    sample(c(1,7), 50, replace = TRUE), # bimodal, all 1s and 7s
    sample(3:5,   50, replace = TRUE) # clustered in the middle
  )
)

# Identical means
demo %>%
  group_by(condition) %>%
  summarise(mean = mean(rating))

# Plot different distributions
ggplot(demo, aes(x = factor(rating), fill = condition)) +
  geom_bar(position = "dodge") +
  scale_fill_viridis_d(option = "D", begin = 0.3, end = 0.75) +
  labs(title = "Same mean, completely different distributions",
       x = "Rating", y = "Count", fill = "Condition") +
  theme_minimal(base_size = 13)

> **The means problem:** Conditions A and B have identical means so a linear model treating
> ratings as continuous would find no difference between them, but their distributions are
> completely different. Condition A is polarised (people either love it or hate it); Condition B
> is neutral (everyone is lukewarm). Ordinal regression can detect this difference.

In [ ]:
# Fit cumulative link mixed model (ordinal regression)
# clmm() from the ordinal package
model_rating <- clmm(rating ~ fsp_position + context + (1 | participant_id) + (1 | item_id),
                     data = acceptability,
                     link = "logit")

summary(model_rating)

> **Reading the output:**
> - **Threshold coefficients:** estimated cutpoints between adjacent rating levels
> - **Fixed effects:** coefficients are on the log-odds (proportional odds) scale so positive = higher ratings
> - **contextmatch:** being in a match context strongly increases the probability of higher ratings
> - **fsp_positionS-FSP-V-O:** the negative coefficient reflects the acceptability penalty for preverbal *only*
---
> **Random effects**
> - Participant variance (3.11, SD = 1.76): substantial variation between participants in their baseline rating tendency
> - Item variance (1.34, SD = 1.16): items also vary in how acceptable they are rated overall, but less than participants
> - Both are on the log-odds scale, not the rating scale directly
---
> **Fixed effects**  
> Reference level: FSP-S-V-O, mismatch  
> All coefficients are on the log-odds scale so positive means higher probability of a higher rating, negative means lower

> - S-FSP-V-O (-0.45, p = .009): preverbal FSP sentences are rated slightly lower than sentence-initial. Significant but the smallest of the position effects
> - S-V-FSP-O (-1.59, p < .001): the largest position effect. Sentences with FSP directly preceding the object are rated considerably lower than sentence-initial
> - S-V-O-FSP (-0.23, p = .182): sentence-final FSP is not significantly different from FSP-initial in terms of acceptability. Non-significant
> - contextmatch (+8.09, p < .001): by far the largest effect. Match contexts receive substantially higher ratings than mismatch contexts, as expected.

In [ ]:
# Compare models with and without FSP position using likelihood ratio test
model_rating_null <- clmm(rating ~ context + (1 | participant_id) + (1 | item_id),
                           data = acceptability, link = "logit")

anova(model_rating_null, model_rating)

> **Interpreting the likelihood ratio test:**
> Adding FSP position significantly improves model fit (check the p-value).
> This tells us FSP position explains variance in acceptability over and above context alone.

---

## 5. Logistic mixed effects model: TVJT accuracy

**Outcome:** `correct` (binary: 0 or 1)  
**Question:** Does FSP position and context predict whether participants correctly judge sentences?

We use `glmer()` with `family = binomial`.

In [ ]:
# Fit logistic mixed effects model
model_acc <- glmer(correct ~ fsp_position + context + (1 | participant_id) + (1 | item_id),
                   data = tvjt,
                   family = binomial(link = "logit"))

summary(model_acc)

> **Singular fit warning**

> This is flagged because the participant random effect variance is essentially zero (= 7.16e-10). The model is telling you that once you account for the fixed effects and item random effects, there is no meaningful variation between participants in their baseline accuracy. This can happen when accuracy is so high across the board that participants barely differ.

> The random effect is genuinely not needed for participants in this model

> What to do: refit without the participant random intercept and compare models:

In [ ]:
model_acc_nopart <- glmer(correct ~ fsp_position + context + (1 | item_id),
                          data = tvjt, family = binomial)

AIC(model_acc, model_acc_nopart)

> The simpler model (without participant random intercept) has a lower AIC by exactly 2 points which is right on the conventional threshold.

> What it means:

> The difference is exactly 2 AIC units, the minimum conventionally considered meaningful. Removing the participant random intercept costs one degree of freedom and saves 2 AIC points.

>This confirms what the singular fit warning already told you: participant random intercept is not contributing meaningful information to the model.

> What to do:  
> Go with the simpler model, i.e., model_acc_nopart.

In [ ]:
summary(model_acc_nopart)

> The singular fit warning is resolved, the model is more parsimonious, and AIC supports the simpler structure.  

> **Reading the output:**
> - Coefficients are in **log-odds**, not probabilities
> - Positive = higher probability of being correct
> - `contextmatch` is large and positive, match conditions are much easier
> - `fsp_positionS-FSP-V-O` is negative, preverbal *only* reduces accuracy
> - z values (not t values) are used for GLMMs and p-values are available by default

> **Random effects**  
Item SD (0.172 log-odds) is small but meaningful. Some items are slightly harder than others to judge correctly. Retaining this random effect is justified and the variance is non-zero



In [ ]:
# Convert log-odds to probabilities using plogis(), the inverse logit function
intercept <- fixef(model_acc_nopart)["(Intercept)"]

# Reference condition
cat("P(correct): FSP-S-V-O, mismatch = ", round(plogis(intercept), 3), "\n")
cat("P(correct): FSP-S-V-O, match = ", round(plogis(intercept +
                                                   fixef(model_acc_nopart)["contextmatch"]), 3), "\n\n")

# FSP position effects -- note: only S-V-FSP-O is significant (p < .001)
cat("P(correct): S-FSP-V-O, mismatch = ", round(plogis(intercept +
                                                   fixef(model_acc_nopart)["fsp_positionS-FSP-V-O"]), 3), "\n")
cat("P(correct): S-FSP-V-O, match = ", round(plogis(intercept +
                                                   fixef(model_acc_nopart)["fsp_positionS-FSP-V-O"] +
                                                   fixef(model_acc_nopart)["contextmatch"]), 3), "\n\n")

cat("P(correct): S-V-FSP-O, mismatch = ", round(plogis(intercept +
                                                   fixef(model_acc_nopart)["fsp_positionS-V-FSP-O"]), 3), "\n")
cat("P(correct): S-V-FSP-O, match = ", round(plogis(intercept +
                                                   fixef(model_acc_nopart)["fsp_positionS-V-FSP-O"] +
                                                   fixef(model_acc_nopart)["contextmatch"]), 3), "\n\n")

cat("P(correct): S-V-O-FSP, mismatch = ", round(plogis(intercept +
                                                   fixef(model_acc_nopart)["fsp_positionS-V-O-FSP"]), 3), "\n")
cat("P(correct): S-V-O-FSP, match = ", round(plogis(intercept +
                                                   fixef(model_acc_nopart)["fsp_positionS-V-O-FSP"] +
                                                   fixef(model_acc_nopart)["contextmatch"]), 3), "\n")

> **Key takeaway:**  
> Converting from log-odds to probabilities makes the results interpretable.  
> The S-V-FSP-O mismatch condition has the lowest accuracy, meaning participants have a harder time correctly judging sentences with *only* directly preceding the object as FALSE.

---

## 6. Linear mixed effects model: Response time

**Outcome:** `rt_ms` (continuous, right-skewed)
**Question:** Does FSP position and context predict response time?

RT data is almost always right-skewed. We need to log-transform before fitting a linear model.

In [ ]:
# Compare raw vs log-transformed RT side by side
p1 <- ggplot(tvjt, aes(x = rt_ms)) +
  geom_histogram(bins = 40, fill = viridis(1, option = "D"), colour = "#2E3440", alpha = 0.85) +
  labs(title = "Raw RT (ms)", x = "RT (ms)", y = "Count") +
  theme_minimal(base_size = 12)

p2 <- ggplot(tvjt, aes(x = log(rt_ms))) +
  geom_histogram(bins = 40, fill = viridis(1, option = "E"), colour = "#2E3440", alpha = 0.85) +
  labs(title = "Log-transformed RT", x = "log(RT)", y = "Count") +
  theme_minimal(base_size = 12)

p1 + p2

> **Notice:** Log-transformation makes the distribution much more symmetric and
> closer to normal. This is what the linear model assumes for its residuals.

In [ ]:
# Add log RT column
tvjt$log_rt <- log(tvjt$rt_ms)

# Fit linear mixed effects model on log RT
model_rt <- lmer(log_rt ~ fsp_position + context + (1 | participant_id) + (1 | item_id),
                 data = tvjt,
                 REML = TRUE)

summary(model_rt)

> **Reading the output:**
> - Coefficients are on the **log ms** scale
> - Negative coefficient for `contextmatch`; match conditions are faster
> - To back-transform to a multiplicative effect on raw RT: `exp(coefficient)`

> **Scaled residuals**

> Roughly symmetric around zero (median -0.02), max of 4.32 suggests a few slow trials but nothing alarming. The positive skew in the max is expected even after log-transformation; a handful of very slow responses will always remain. Could remove outliers before modelling, but sometimes they are informative.

> **Random effects**

> - Participant SD (0.117 log-ms): meaningful variation between participants in baseline RT. Some people are just consistently faster or slower than others. Much larger than item variance.
> - Item SD (0.025 log-ms): very small. Items are well controlled and barely differ in how quickly they elicit responses
> - Residual SD (0.447 log-ms): the remaining within-participant, within-item variability after accounting for fixed effects and random effects


In [ ]:
# Back-transform coefficients for reporting
# exp(coefficient) gives the multiplicative effect on raw RT
cat("Multiplicative RT effects (relative to reference condition):\n\n")

fe <- fixef(model_rt)
for (nm in names(fe)) {
  cat(sprintf("  %-35s exp(b) = %.3f\n", nm, exp(fe[nm])))
}

> **Interpreting multiplicative effects:**
> - exp(b) = 1.0 means no effect on RT
> - exp(b) = 1.1 means RT is 10% longer
> - exp(b) = 0.9 means RT is 10% shorter

> **Fixed effects**  
> Reference level: FSP-S-V-O, mismatch
> Coefficients are on the log-ms scale. To interpret as multiplicative effects on raw RT use exp().

> - Intercept (6.601, p < .001): predicted log-RT for the reference condition; exp(6.601) = 735ms predicted RT for FSP-S-V-O mismatch
> - S-FSP-V-O (+0.017, p = .622): not significant; preverbal FSP is not slower than FSP-initial; exp(0.017) = 1.02, essentially no difference
> - S-V-FSP-O (+0.077, p = .027): significant; FSP directly preceding the object is slower; exp(0.077) = 1.08 where (1.0801 - 1) × 100 ≈ 8.01%, thus,  about 8% longer RT than the reference condition; exp(6.601 + 0.077) = 794ms predicted RT
> - S-V-O-FSP (+0.035, p = .307): not significant; sentence-final FSP is not significantly slower than FSP-initial
> - contextmatch (-0.130, p < .001): significant and negative, match contexts are faster; exp(-0.130) = 0.88, about 12% shorter RT than mismatch; exp(6.601 - 0.130) = 647ms predicted RT for FSP-S-V-O match

In [ ]:
# Final assumption check -- residuals for RT model
par(mfrow = c(1, 2))
plot(model_rt, main = "Residuals vs. Fitted: RT model")
qqnorm(resid(model_rt), main = "Q-Q plot: RT model residuals")
qqline(resid(model_rt), col = viridis(1, option = "A"))
par(mfrow = c(1, 1))

---
# -------------------- Synthetic study 2: Corpus study --------------------

## 7. Modelling count data: FSP occurrence in corpus data

**Outcome:** `count` (non-negative integer; number of FSP occurrences per document per position)  

**Question:** Does FSP position frequency vary across corpus types (register)?

**Dataset:** 160 documents across 8 corpora (2 per corpus type):
- **Newspaper**: written formal
- **Social media written**: written informal
- **Legal proceedings**: spoken formal
- **Social media spoken**: spoken informal

Each document contributes 4 rows, one count per FSP position. Total: 640 rows.

### 7.1 Load the data

In [ ]:
# Load corpus dataset
corpus <- read.csv("https://raw.githubusercontent.com/lhracs/CLA2026_crash_course/refs/heads/main/corpus_synth.csv",
                   stringsAsFactors = FALSE)

# Set factor levels
corpus$corpus_type  <- factor(corpus$corpus_type,
                               levels = c("newspaper","social_media_written",
                                          "legal","social_media_spoken"))
corpus$fsp_position <- factor(corpus$fsp_position,
                               levels = c("FSP-S-V-O","S-FSP-V-O","S-V-FSP-O","S-V-O-FSP"))

cat("Loaded coprus data:", nrow(corpus), "rows\n")
head(corpus)

### 7.2 Inspect the data

In [ ]:
corpus %>%
  group_by(corpus_type) %>%
  dplyr::slice(1) %>%
  dplyr::select(corpus_type, corpus, word_count, excerpt)

### 7.3 Visualize counts by corpus type and FSP position

In [ ]:
# Mean FSP count per document by corpus type and position
corpus_summary <- corpus %>%
  group_by(corpus_type, fsp_position) %>%
  summarise(mean_count = mean(count), se = sd(count) / sqrt(n()), .groups = "drop")

ggplot(corpus_summary, aes(x = corpus_type, y = mean_count, fill = fsp_position)) +
  geom_col(position = position_dodge(0.75), width = 0.7) +
  geom_errorbar(aes(ymin = mean_count - se, ymax = mean_count + se),
                position = position_dodge(0.75), width = 0.25) +
  scale_fill_viridis_d(option = "D") +
  labs(
    title = "Mean FSP occurrence count per document by corpus type",
    subtitle = "Error bars = +/- 1 SE",
    x = "Corpus type",
    y = "Mean count per document",
    fill = "FSP position"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top",
        axis.text.x = element_text(angle = 15, hjust = 1))

> **What to notice:**
> - Raw counts are not directly comparable across corpus types; legal documents are much longer than social media posts
> - This is why we need an **offset** for word count when modelling
> - FSP-S-V-O dominates in formal registers (newspaper, legal); S-V-O-FSP is more common in informal spoken
> - The pattern seems to be meaningful; formal registers may favour clause-initial focus marking

### 7.3 Check for overdispersion

In [ ]:
# Check the mean-variance relationship
# For Poisson: variance should equal the mean
# For overdispersed data: variance >> mean

dispersion_check <- corpus %>%
  group_by(fsp_position) %>%
  summarise(
    mean_count = mean(count),
    var_count  = var(count),
    ratio      = var(count) / mean(count),
    .groups = "drop"
  )

print(dispersion_check)

> **What to notice:**
> - The variance/mean ratio should be approximately 1 for Poisson data
> - Ratios well above 1 indicate overdispersion; variance is much larger than the mean
> - This is extremely common in corpus count data
> - We need **negative binomial regression** instead of Poisson

In [ ]:
# Fit a Poisson model, then check residuals
model_poisson <- glmer(count ~ fsp_position + corpus_type + offset(log(word_count)) +
                      (1 | corpus), data   = corpus, family = poisson(link = "log"))

# Quick overdispersion test; residual deviance / df should be ~1
cat("Residual deviance:", deviance(model_poisson), "\n")
cat("Residual degrees of freedom:", df.residual(model_poisson), "\n")
cat("Ratio (should be ~1 for Poisson):", round(deviance(model_poisson) / df.residual(model_poisson), 2), "\n")

> The ratio is asking, on average, how much unexplained variance is there per observation, after accounting for model complexity. If the ratio is much greater than 1, the Poisson model is a poor fit.
> The correct model for overdispersed count data is **negative binomial regression**.

In [ ]:
# Fit negative binomial model
# Note: glmer.nb() from lme4 handles mixed effects negative binomial
model_nb <- glmer.nb(count ~ fsp_position + corpus_type + offset(log(word_count)) +
                    (1 | corpus), data = corpus)

summary(model_nb)

> **Reading the output:**
> - Coefficients are on the **log scale**; exponentiate for interpretable rate ratios
> - **Intercept:** log rate for the reference condition (FSP-S-V-O, newspaper)
> - **fsp_positionS-FSP-V-O:** how much the rate differs from FSP-S-V-O in the same corpus type
> - **corpus_typelegal:** how much the rate differs from newspaper for the same FSP position
> - **offset(log(word_count)):** controls for document length; we are modelling rate, not raw count
> - **(1 | corpus):** random intercept for corpus accounts for within-corpus clustering

> **Random effects**  
> Corpus SD (0.323 log-count): meaningful variation between the 8 corpora in their baseline FSP rate after accounting for corpus type and FSP position. Even within the same corpus type, individual corpora differ. This justifies including the random effect.

In [ ]:
# Exponentiate coefficients for rate ratios
cat("Rate ratios (exp(b)) relative to reference (FSP-S-V-O, newspaper):\n\n")
fe <- fixef(model_nb)
for (nm in names(fe)) {
  cat(sprintf("  %-45s exp(b) = %.3f\n", nm, exp(fe[nm])))
}

> **Interpreting rate ratios:**
> - exp(b) = 1.0 means no difference in rate
> - exp(b) = 2.0 means twice the rate of the reference condition
> - exp(b) = 0.5 means half the rate of the reference condition
>
> For example, if S-V-O-FSP has exp(b) = 0.3 relative to FSP-S-V-O in newspaper,
> it occurs at about 30% of the rate. This result is consistent with formal registers disfavouring post-verbal focus sensitive particles.

> **Fixed effects**
> Reference level: FSP-S-V-O, newspaper
> All coefficients are log rate ratios, so we have to exponentiate for interpretation

> - Intercept (-4.651, p < .001): log rate for FSP-S-V-O in newspaper per word; exp(-4.651) = 0.0095, so about 9.5 occurrences per 1000 words in the reference condition
> - S-FSP-V-O (-0.353, p < .001): significant; exp(-0.353) = 0.703, so preverbal FSP occurs at about 70% of the rate of FSP-initial in the same corpus type, i.e., at about 30% lower rate
> - S-V-FSP-O (-0.433, p < .001): significant; exp(-0.433) = 0.649, so FSP directly preceding object occurs at about 65% of the FSP-initial rate, i.e., at about 35% lower rate
> - S-V-O-FSP (-0.833, p < .001): significant and the largest position effect;exp(-0.833) = 0.435, so sentence-final FSP occurs at less than half the rate of sentence-initial, i.e., about 57% lower rate;  sentence-final position is strongly dispreferred relative to sentence-initial across all corpus types
> - social_media_written (+0.367, p = .303): not significant; social media written is not significantly different from newspaper in FSP rate
> - legal (-0.053, p = .874): not significant;  legal proceedings are not significantly different from newspaper in FSP rate
> - social_media_spoken (+0.570, p = .096) not significant; social media spoken shows a trend toward higher FSP rate than newspaper but does not reach significance

In [ ]:
# Compare Poisson vs. negative binomial fit using AIC
cat("Poisson AIC:", AIC(model_poisson), "\n")
cat("Negative binomial AIC:", AIC(model_nb), "\n")
cat("\nLower AIC = better fit. A difference > 2 is meaningful.\n")

> **AIC comparison:**
> The negative binomial model should have a substantially lower AIC than the Poisson model,
> confirming that accounting for overdispersion gives a better fit to the data.
> This is the standard way to justify choosing negative binomial over Poisson in a methods section.

---

## Summary

We have worked through five analyses covering some of the main model families used in linguistic research:

| Task | Outcome | Model | R function |
|------|---------|-------|------------|
| Production | Peak F0 (Hz) | Linear mixed effects | `lmer()` |
| Acceptability | Rating (1-7) | Ordinal regression | `clmm()` |
| TVJT | Accuracy (0/1) | Logistic mixed effects | `glmer()` |
| TVJT | Response time (ms) | Linear mixed effects on log RT | `lmer()` |
| Corpus | FSP occurrence count | Negative binomial mixed effects | `glmer.nb()` |

---

### Key methodological takeaways

**Your outcome variable drives everything**
- Every modelling decision started with asking what type of variable your outcome is
- Getting this wrong may produce results that look fine, but in fact, are not

**Always visualize before you model**
- The RT histogram revealed right skew before we fitted anything which is why we log-transformed
- Rating distributions showed discrete clustering which is why we used `clmm()` not `lmer()`
- Plots can make the modelling decisions obvious before running a single model

**It is important to consider random effects**
- Models included random intercepts for the appropriate grouping factors
- The logistic model flagged a singular fit, which led us to simplify and produce a better model

**Model comparison is part of the analysis**
- Likelihood ratio test to justify the ordinal model over the null
- AIC to confirm choice between Poisson and negative binomial
- AIC to resolve the singular fit in the logistic model
- Model selection is built into the pipeline, not an afterthought

**Multiple tasks reveal a richer picture**
- FSP position affected F0, acceptability, accuracy, and RT, but not always the same positions (based on these synthetic data)
- S-V-FSP-O was the hardest condition in accuracy and RT
- S-FSP-V-O showed an acceptability penalty but not an accuracy penalty
- A single task and a single model would have missed these nuances

**Assumption checking leads to better decisions**
- Residuals and Q-Q plots for linear models
- Dispersion test for count data
- Singular fit check for all mixed models

---

*Workshop materials by Lindsay Hracs for CLA 2026 Crash Course*  
*Contact: lindsay[dot]hracs[at]ucalgary[dot]ca*